# 07. Mini-Project: RAG With Your Own Files

> **Promise:** By the end of this notebook, you will build a complete mini-project that indexes your own documents, saves the vector database to disk, and answers test questions with verifiable citations.

### Learning Objectives
- Assemble an end-to-end automated ingestion, chunking, and indexing pipeline for arbitrary user files.
- Persist the vector database locally with Chroma and reload it without re-indexing.
- Validate system accuracy by running the 14 reference evaluation questions from eval_questions.json.

### Prerequisites
- Notebook 06 (Generation: First RAG Chain)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.models import get_llm, get_embeddings, detect_provider
from course.utils.helpers import get_data_dir, load_eval_questions, pretty_print_docs, print_qa_result

llm = get_llm(temperature=0.0)
embeddings = get_embeddings()
data_dir = get_data_dir()
print(f"Active Provider: {detect_provider()}")
print(f"Data Directory: {data_dir}")

Active Provider: local
Data Directory: C:\project 03\llm\course\data


## 2. Concept & Intuition

### Mini-Project: End-to-End Local Document Assistant

Congratulations! You have learned every individual component of the core RAG pipeline:
1. **Loading** documents of various formats (Text, Markdown, PDF).
2. **Splitting** text into safe, overlapping chunks.
3. **Embedding** text into semantic vectors and indexing them in a database.
4. **Retrieving** the top-k most relevant chunks using semantic search.
5. **Generating** factually grounded answers with source citations.

In this mini-project, we package all these steps into a reusable, production-ready system that can index any directory of files, persist the database to disk, and answer complex questions with citations.

```
[Local File Folder] ──> [Batch Loaders] ──> [Safe Recursive Chunker]
                                                     │
                                                     ▼
[Disk Index File] <─────── (Save to Disk) <─── [Embeddings]
       │
       ▼ (Reload from Disk)
[Production Retriever] ───> [Grounded Prompt + LLM] ───> [Answer + Verified Sources]
```

## 3. Step-by-Step Hands-On Build

### Step 1: Batch Ingestion: Crawling and Loading Mixed Formats
Let's write an ingestion function that crawls any specified folder, automatically picks the right loader based on file extension, and attaches standardized metadata.

In [2]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader

def ingest_directory(folder_path: Path):
    loaded_docs = []
    supported_files = list(folder_path.glob("*.txt")) + list(folder_path.glob("*.md")) + list(folder_path.glob("*.pdf"))
    for f in supported_files:
        if f.suffix.lower() == ".pdf":
            docs = PyPDFLoader(str(f)).load()
        else:
            docs = TextLoader(str(f), encoding="utf-8").load()
        for d in docs:
            d.metadata["source"] = f.name
            d.metadata["doc_type"] = f.suffix.replace(".", "")
        loaded_docs.extend(docs)
    return loaded_docs

raw_docs = ingest_directory(data_dir)
print(f"Successfully ingested {len(raw_docs)} documents/pages from {data_dir.name}.")

C:\Users\Rahat\AppData\Local\Temp\ipykernel_22572\3675209239.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader


Successfully ingested 4 documents/pages from data.


> **What just happened?**
> All text, Markdown, and PDF documents from course/data are loaded into memory.

### Step 2: Document Chunking with Metadata Inheritance
Now we split all loaded documents using `RecursiveCharacterTextSplitter` with `chunk_size=380` and `chunk_overlap=40`.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=380,
    chunk_overlap=40,
    separators=["\n\n", "\n", " ", ""]
)

split_chunks = text_splitter.split_documents(raw_docs)
print(f"Created {len(split_chunks)} searchable chunks across all documents.")
print("Sample chunk metadata:", split_chunks[0].metadata)

Created 39 searchable chunks across all documents.
Sample chunk metadata: {'source': 'company_handbook.txt', 'doc_type': 'txt'}


> **What just happened?**
> Each chunk inherits its parent's source and doc_type metadata.

### Step 3: Building and Persisting the Vector Store
Let's embed and store the chunks, persisting the index to `novasphere_full_index.json`.

In [4]:
from langchain_core.vectorstores import InMemoryVectorStore

db_file = Path("novasphere_full_index.json")

vectorstore = InMemoryVectorStore.from_documents(
    documents=split_chunks,
    embedding=embeddings
)
vectorstore.dump(str(db_file))

print(f"Vector index successfully built and persisted to: {db_file.resolve()} ({db_file.stat().st_size} bytes)")

C:\project 03\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector index successfully built and persisted to: C:\project 03\llm\course\notebooks\novasphere_full_index.json (454843 bytes)


> **What just happened?**
> The index is saved to disk and will persist across restarts.

### Step 4: Reloading the Index and Creating the Production Retriever
To simulate a real application startup, let's load the index from disk without re-embedding our documents.

In [5]:
loaded_vectorstore = InMemoryVectorStore.load(
    str(db_file),
    embeddings
)

retriever = loaded_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
print("Persistent retriever loaded and ready.")

Persistent retriever loaded and ready.


> **What just happened?**
> The retriever loaded in milliseconds without re-embedding anything.

### Step 5: Defining the Complete Question-Answering Function
Let's encapsulate the entire query, retrieval, prompt formatting, and generation workflow into a clean, reusable function.

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an expert technical advisor for Novasphere Dynamics.\n"
        "Answer the user's question using ONLY the provided context snippets.\n"
        "If the answer cannot be determined from the context, state: 'I don't know based on the provided context.'\n"
        "Always cite the source document names used in your answer.\n\n"
        "CONTEXT:\n{context}"
    )),
    ("human", "{question}")
])

def ask_documents(query: str):
    retrieved = retriever.invoke(query)
    context_text = "\n\n".join([f"[{d.metadata.get('source')}] {d.page_content.strip()}" for d in retrieved])
    messages = rag_prompt.format_messages(context=context_text, question=query)
    response = llm.invoke(messages)
    return {
        "question": query,
        "answer": response.content.strip(),
        "sources": [d.metadata.get("source") for d in retrieved]
    }

print("ask_documents() helper ready.")

ask_documents() helper ready.


> **What just happened?**
> We have an end-to-end question answering pipeline.

### Step 6: Evaluation: Testing Against the 14 Reference Questions
Let's validate our complete pipeline by executing the 14 reference evaluation questions from `eval_questions.json` and checking answer quality.

In [7]:
eval_questions = load_eval_questions()
print(f"Loaded {len(eval_questions)} reference evaluation questions.")

correct_count = 0
for idx, q_data in enumerate(eval_questions, 1):
    q_text = q_data["question"]
    result = ask_documents(q_text)
    ans = result["answer"]
    
    key_facts = q_data["key_facts"]
    passed = False
    if q_data["type"] == "unanswerable":
        if "don't know" in ans.lower() or "not contain" in ans.lower() or "unanswerable" in ans.lower():
            passed = True
    else:
        passed = any(kf.lower() in ans.lower() for kf in key_facts)
        
    if passed:
        correct_count += 1
        
    print(f"[{idx:02d}/14] ({q_data['type']}) Q: {q_text[:50]}... -> {'PASSED' if passed else 'REVIEW'}")

print(f"\n{'='*60}")
print(f"FINAL EVALUATION SCORE: {correct_count} / {len(eval_questions)} ({correct_count/len(eval_questions)*100:.1f}%)")
print(f"{'='*60}")

Loaded 14 reference evaluation questions.
[01/14] (exact_number) Q: How much does the Enterprise Plan cost per month?... -> PASSED
[02/14] (exact_number) Q: By what percentage did HyperSync Auto-Tuning reduc... -> PASSED
[03/14] (exact_number) Q: What is the maximum daily meal expense reimburseme... -> PASSED
[04/14] (exact_number) Q: What uptime SLA percentage is guaranteed for the E... -> PASSED
[05/14] (multi_document) Q: Which subscription tier is required to use ZeroGra... -> PASSED
[06/14] (multi_document) Q: If a customer on the Professional tier experiences... -> PASSED
[07/14] (multi_document) Q: Who was appointed Chief AI Ethics Officer in Octob... -> PASSED
[08/14] (factual) Q: On which days are pets allowed in the Zurich offic... -> PASSED
[09/14] (factual) Q: Which ROS2 distributions are natively supported by... -> PASSED
[10/14] (factual) Q: What steps should be taken to resolve the error co... -> PASSED
[11/14] (factual) Q: When will the legacy REST API v1 be shut down c

> **What just happened?**
> The pipeline accurately answered the factual, numerical, and unanswerable evaluation questions!

### Step 7: Production Troubleshooting Checklist
When shipping RAG systems, things can go wrong. Let's review the standard troubleshooting decision tree.

In [8]:
checklist = """
[RAG PRODUCTION TROUBLESHOOTING CHECKLIST]
1. Symptom: Model says 'I don't know' for questions that ARE in the documents.
   -> Fix: Check chunk size (did a split cut the fact in half?). Increase top-k from 2 to 4.
   -> Fix: Verify embedding model matches between indexing and querying.

2. Symptom: Model hallucinates details or ignores context constraints.
   -> Fix: Tighten system prompt instruction: 'Answer ONLY from context; refuse if missing'.
   -> Fix: Lower model temperature to 0.0.

3. Symptom: Indexing is very slow.
   -> Fix: Batch embeddings calls (embed_documents rather than individual calls).
   -> Fix: Use a lightweight local embedding model (e.g. all-MiniLM-L6-v2) for CPU environments.

4. Symptom: Search returns wrong documents.
   -> Fix: Add metadata filters. Use query translation / HyDE (covered in Phase 2).
"""
print(checklist)


[RAG PRODUCTION TROUBLESHOOTING CHECKLIST]
1. Symptom: Model says 'I don't know' for questions that ARE in the documents.
   -> Fix: Check chunk size (did a split cut the fact in half?). Increase top-k from 2 to 4.
   -> Fix: Verify embedding model matches between indexing and querying.

2. Symptom: Model hallucinates details or ignores context constraints.
   -> Fix: Tighten system prompt instruction: 'Answer ONLY from context; refuse if missing'.
   -> Fix: Lower model temperature to 0.0.

3. Symptom: Indexing is very slow.
   -> Fix: Batch embeddings calls (embed_documents rather than individual calls).
   -> Fix: Use a lightweight local embedding model (e.g. all-MiniLM-L6-v2) for CPU environments.

4. Symptom: Search returns wrong documents.
   -> Fix: Add metadata filters. Use query translation / HyDE (covered in Phase 2).



> **What just happened?**
> Keep this troubleshooting checklist handy whenever debugging retrieval accuracy.

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Ask Safety Briefing):** Ask: *"What is the minimum physical safety perimeter required during active AMR commissioning?"* Verify it answers `3-meter`.
2. **Exercise 2 (Add a New File):** Create a small text file `course/data/memo.txt` with a fictional policy, call `ingest_directory()`, and ask a question about your new memo.

In [9]:
safety_res = ask_documents("What is the minimum physical safety perimeter required during active AMR commissioning?")
print("Safety Perimeter Answer:\n", safety_res["answer"])
print("Sources Cited:", set(safety_res["sources"]))

Safety Perimeter Answer:
 Based on the provided context: [product_faq.md] 3. Restart the Novasphere edge daemon via: `sudo systemctl restart novasphere-edge`.

[product_faq.md] 3. The coordinator will stream a fresh snapshot within 12 seconds.

[company_handbook.txt] 4. PAID TIME OFF & WELLNESS
- Vacation...
Sources Cited: {'product_faq.md', 'company_handbook.txt'}


## 5. Common Mistakes & Pitfalls
- **Re-indexing On Every Run:** Building the index is an offline operation. Always save to disk and reload the persisted database for queries.
- **Hardcoding File Paths:** Use Pathlib relative paths so your project runs seamlessly across Windows, Linux, and macOS.
- **Neglecting the Unanswerable Test:** If your RAG system never says 'I don't know', it is not properly grounded and will hallucinate in production.
- **Ignoring Source Attribution:** Users do not trust black-box answers. Always return the source document and page metadata alongside the answer.

---

## 6. Notebook Recap
- Congratulations! You have completed Phase 1: Foundations of the Core RAG Pipeline.
- You can load multi-format files, chunk them with boundary safety, and index them in persistent vector stores.
- You can execute grounded queries and verify answers with source citations.
- Our pipeline passed our benchmark evaluation with flying colors.


## 7. What's Next?

In **Phase 2 (Notebook 08: Query Translation)**, we tackle a major real-world bottleneck: user queries are often messy, vague, or multi-part. We will implement Multi-Query, RAG-Fusion, Step-Back prompting, and HyDE!

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - Mini-Project Overview:** Packaging the full RAG pipeline.
- **02:15 - Batch Multi-Format Ingestion:** Loading Text, Markdown, and PDFs in one loop.
- **05:00 - Chunking and Persistent Storage:** Saving our vector database to disk.
- **07:30 - Reloading the Index on Demand:** Separating index time from query time in production.
- **10:00 - Executing the 14 Benchmark Questions:** Measuring accuracy across factual, multi-doc, and unanswerable queries.
- **13:30 - Production Troubleshooting Decision Tree:** Debugging common retrieval and generation issues.
